# Kiểm tra bộ căn chỉnh E — không huấn luyện

62 cặp / 59 ảnh: đổi quan hệ12, đổi vật thể12, đổi người38.
Cả hai GPU cùng suy luận một checkpoint E; không chạy hai nhánh huấn luyện.
Loại toàn bộ ảnh trong10k ảnh train pilot E. Các ảnh vẫn thuộc COCO train/restval và đã xuất hiện trong tiền huấn luyện caption; đây là chẩn đoán, không phải test độc lập.

Chạy ngay sau notebook E trên cùng session hoặc thêm saved output/checkpoint E vào notebook mới.
**Chỉnh CHECKPOINT bên dưới đúng đường dẫn checkpoint.pth của E**. Không dùng checkpoint SCST gốc/A.
Giữ đúng checkpoint SHA651f411f...; notebook kiểm tra hash trước khi tải mô hình.
Chọn T4x2, bật Internet, bạn tự Save & Run All. Không cần raw ảnh, prompt cache, V-COCO hay legacy prompt dataset.

In [ ]:
print('FROZEN E MATCHER PROBE: START — NO TRAINING', flush=True)
import os
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', CUBLAS_WORKSPACE_CONFIG=':4096:8',
                  HF_HUB_DISABLE_XET='1', TOKENIZERS_PARALLELISM='false', PYTHONHASHSEED='42',
                  OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
from pathlib import Path
CHECKPOINT = Path('/kaggle/working/semantic_evidence_pilot_E/checkpoint.pth')
VISUAL_CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
OUTPUT = Path('/kaggle/working/semantic_evidence_matcher_probe')
# Nếu mở notebook mới: thay CHECKPOINT bằng đường dẫn chính xác từ Input của output E đã lưu.
assert CHECKPOINT.is_file(), f'Missing E checkpoint: {CHECKPOINT}. Copy its actual Input path; do not use original/A.'
assert VISUAL_CACHE.is_dir(), VISUAL_CACHE
assert not OUTPUT.exists(), f'Use a fresh OUTPUT: {OUTPUT}'


In [ ]:
import json, subprocess, sys, hashlib
REPO = Path('/tmp/Image_Captioning-semantic-evidence-probe')
def run(args, cwd=None):
    print('Running:', ' '.join(map(str,args)), flush=True)
    subprocess.run(list(map(str,args)), cwd=cwd, check=True)
if not REPO.exists(): run(['git','clone','--no-checkout','https://github.com/Supzxjee/Image_Captioning.git',REPO])
assert (REPO/'.git').is_dir(), REPO
run(['git','fetch','--depth','1','origin','8242105fdb0099700e9e8d82a6e098c5ce3ed700'],cwd=REPO)
run(['git','checkout','--detach','8242105fdb0099700e9e8d82a6e098c5ce3ed700'],cwd=REPO)
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy'])
PROBE = REPO/'frozen_e_probe'; PROBE.mkdir(exist_ok=True)
os.environ['PYTHONPATH'] = str(REPO) + os.pathsep + str(PROBE)

SOURCES = {'run_probe.py': '"""Read-only E matcher probe. Two independent inference workers; no optimizer or training."""\nimport argparse\nfrom collections import defaultdict\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom probe_data import digest, identity, summarize\n\nCLIP = \'openai/clip-vit-base-patch16\'\nREVISION = \'57c216476eefef5ab752ec549e440a49ae4ae5f3\'\n\n\ndef load_manifest(cfg):\n    for name, expected in cfg[\'source_sha256\'].items():\n        if digest(Path(__file__).parent/name) != expected:\n            raise ValueError(f\'Probe source changed: {name}\')\n    manifest = json.loads(Path(cfg[\'manifest\']).read_text(encoding=\'utf-8\'))\n    if digest(cfg[\'manifest\']) != cfg[\'manifest_sha256\'] or identity(manifest[\'rows\']) != manifest[\'rows_sha256\']:\n        raise ValueError(\'Probe manifest changed\')\n    assert manifest[\'alignment_train_overlap\'] == 0 and not manifest[\'test_used\']\n    assert manifest[\'excluded_pilot_train_images\'] == 10000\n    assert len({r[\'case_id\'] for r in manifest[\'rows\']}) == len(manifest[\'rows\'])\n    return manifest\n\n\ndef worker(config, rank):\n    import numpy as np\n    import torch\n    from transformers import CLIPTextModel, AutoTokenizer\n    from captioning.visual_cache import VisualCache\n    from captioning.scst_runtime import configure_strict_runtime, runtime_manifest, model_digest\n    from pilot_model import EvidenceMemory\n    cfg = json.loads(Path(config).read_text())\n    manifest = load_manifest(cfg)\n    if digest(cfg[\'checkpoint\']) != manifest[\'checkpoint_sha256\']:\n        raise ValueError(\'Expected the exact completed E checkpoint, not A or the original SCST checkpoint\')\n    configure_strict_runtime(True)\n    torch.cuda.set_device(rank)\n    torch.manual_seed(42)\n    device = f\'cuda:{rank}\'\n    checkpoint = torch.load(cfg[\'checkpoint\'], map_location=\'cpu\', weights_only=True)\n    assert checkpoint[\'format\'] == \'semantic_evidence_pilot_v1\' and checkpoint[\'arm\'] == \'E\'\n    assert checkpoint[\'completed_updates\'] == 1250\n    assert checkpoint[\'protocol_sha256\'] == manifest[\'checkpoint_protocol_sha256\']\n    assert checkpoint[\'cohort_sha256\'] == manifest[\'checkpoint_cohort_sha256\']\n    module = EvidenceMemory().to(device).eval().requires_grad_(False)\n    state = {k.removeprefix(\'extra.\'):v for k,v in checkpoint[\'model_state_dict\'].items() if k.startswith(\'extra.\')}\n    module.load_state_dict(state, strict=True)\n    del checkpoint, state\n    initial_digest = model_digest(module)\n    encoder = CLIPTextModel.from_pretrained(CLIP, revision=REVISION).to(device).eval().requires_grad_(False)\n    tokenizer = AutoTokenizer.from_pretrained(CLIP, revision=REVISION)\n    image_ids = sorted({r[\'coco_id\'] for r in manifest[\'rows\']})[rank::2]\n    selected = set(image_ids)\n    rows = [r for r in manifest[\'rows\'] if r[\'coco_id\'] in selected]\n    texts = sorted({c[\'text\'] for row in rows for c in row[\'candidates\']})\n    embeddings = {}\n    with torch.inference_mode():\n        for start in range(0, len(texts), 64):\n            batch = texts[start:start+64]\n            token = tokenizer(batch, padding=True, truncation=True, max_length=77, return_tensors=\'pt\').to(device)\n            for text, vector in zip(batch, encoder(**token).pooler_output.float()):\n                embeddings[text] = vector.detach().clone()\n    del encoder\n    cache = VisualCache([cfg[\'visual_cache\']], id_key=\'coco_id\')\n    cache.require_ids(image_ids, \'locked evidence diagnostic\')\n    groups = defaultdict(list)\n    for row in rows: groups[row[\'coco_id\']].append(row)\n    results = []\n    try:\n        # Read in source-row order without copying the multi-GB visual cache to /tmp.\n        for count, image_id in enumerate(sorted(image_ids, key=lambda i:cache.index[i]), 1):\n            image = torch.from_numpy(cache.read(image_id)[1:]).to(device)\n            assert image.shape == (196,768) and torch.isfinite(image).all()\n            for row in groups[image_id]:\n                text = torch.stack([embeddings[c[\'text\']] for c in row[\'candidates\']])\n                boxes = torch.tensor([c[\'bounds\'] for c in row[\'candidates\']], device=device)\n                patches = image[None].expand(2,-1,-1)\n                roles = torch.full((2,), 2, dtype=torch.long, device=device)\n                scores = {}\n                with torch.inference_mode():\n                    for method in (\'boxed\', \'global_view\', \'zero_visual\'):\n                        b, p = boxes, patches\n                        if method == \'global_view\':\n                            b = torch.zeros_like(boxes); b[:,2,2:] = 1\n                        if method == \'zero_visual\': p = torch.zeros_like(patches)\n                        _, values, _ = module.attend(p, text, roles, b)\n                        assert torch.isfinite(values).all()\n                        scores[method] = values.cpu().tolist()\n                results.append(dict(row, scores=scores))\n            if count == 1 or count % 10 == 0 or count == len(image_ids):\n                print(f\'GPU{rank}: {count}/{len(image_ids)} images; {len(results)} pairs\', flush=True)\n    finally:\n        cache.close()\n    assert model_digest(module) == initial_digest\n    runtime = runtime_manifest()\n    runtime[\'worker_gpu\'] = dict(rank=rank, name=torch.cuda.get_device_name(rank))\n    payload = dict(rank=rank, records=results, no_training=True, parameters_unchanged=True,\n                   module_state_sha256=initial_digest, runtime=runtime)\n    out = Path(cfg[\'output\'])\n    temporary = out/f\'rank{rank}.json.partial\'\n    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding=\'utf-8\')\n    temporary.replace(out/f\'rank{rank}.json\')\n\n\ndef run(config):\n    cfg = json.loads(Path(config).read_text())\n    manifest = load_manifest(cfg)\n    out = Path(cfg[\'output\'])\n    if out.exists(): raise FileExistsError(f\'Use a fresh output: {out}\')\n    out.mkdir(parents=True)\n    logs, workers = [], []\n    try:\n        for rank in (0,1):\n            log = (out/f\'worker{rank}.log\').open(\'w\',encoding=\'utf-8\'); logs.append(log)\n            workers.append(subprocess.Popen([sys.executable,\'-u\',__file__,\'--config\',str(Path(config).resolve()),\'--rank\',str(rank)], stdout=log, stderr=subprocess.STDOUT))\n        last = 0\n        while any(p.poll() is None for p in workers):\n            if any(p.poll() not in (None, 0) for p in workers):\n                raise RuntimeError(\'Inference worker failed; inspect worker logs\')\n            if time.monotonic()-last >= 10:\n                for rank in (0,1):\n                    lines = (out/f\'worker{rank}.log\').read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()\n                    print(f\'Worker{rank}: \'+(lines[-1] if lines else \'loading pinned checkpoint/text encoder\'), flush=True)\n                last = time.monotonic()\n            time.sleep(1)\n        if any(p.returncode for p in workers): raise RuntimeError(\'Worker failed\')\n    finally:\n        for process in workers:\n            if process.poll() is None: process.terminate(); process.wait()\n        for log in logs: log.close()\n        for rank in (0,1):\n            path=out/f\'worker{rank}.log\'\n            if path.exists(): print(path.read_text(encoding=\'utf-8\',errors=\'replace\')[-2000:], flush=True)\n    shards = [json.loads((out/f\'rank{rank}.json\').read_text()) for rank in (0,1)]\n    assert shards[0][\'module_state_sha256\'] == shards[1][\'module_state_sha256\']\n    rows = sorted([r for s in shards for r in s[\'records\']], key=lambda r:(r[\'task\'],r[\'case_id\']))\n    assert [{k:v for k,v in r.items() if k!=\'scores\'} for r in rows] == manifest[\'rows\']\n    summary = dict(status=\'FROZEN_MATCHER_DIAGNOSTIC_COMPLETE\', metrics=summarize(rows),\n        checkpoint_sha256=manifest[\'checkpoint_sha256\'], manifest_sha256=cfg[\'manifest_sha256\'],\n        no_training=True, parameters_unchanged=True, test_used=False,\n        limitations=[manifest[\'caption_pretraining_overlap\'], manifest[\'scope\'],\n            \'Holding/riding labels follow V-COCO ontology; not an exhaustive universal relation truth.\',\n            \'Cosine pair ranking is not calibrated confidence or caption-quality evidence.\',\n            \'GT/human boxes differ from UPT detector boxes used during E training.\',\n            \'Zero visual features are an out-of-distribution lexical/bias control, not a fair standalone model.\',\n            \'Global-view removes box membership/masks; unchanged scores may reflect patch/box resolution.\',\n            \'Reused selected diagnostic, limited actions; bootstrap samples images, not training seeds.\'])\n    for name, value in [(\'scores.json\',rows),(\'summary.json\',summary)]:\n        (out/name).write_text(json.dumps(value, indent=2, allow_nan=False),encoding=\'utf-8\')\n    shutil.copyfile(cfg[\'manifest\'], out/\'probe_manifest.json\')\n    shutil.copyfile(config, out/\'config.json\')\n    source_dir = out/\'source\'; source_dir.mkdir()\n    for name in cfg[\'source_sha256\']:\n        shutil.copyfile(Path(__file__).parent/name, source_dir/name)\n    print(json.dumps(summary,indent=2),flush=True)\n    archive=shutil.make_archive(str(out)+\'_review\',\'zip\',out)\n    print(\'Download:\',archive,flush=True)\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--config\',required=True);p.add_argument(\'--rank\',type=int,choices=(0,1))\n    a=p.parse_args()\n    if a.rank is None: run(a.config)\n    else: worker(a.config,a.rank)\n', 'probe_data.py': '"""Verified pair construction and tie-aware summaries; no model or training imports."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport numpy as np\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef bounds(subject, obj):\n    s, o = np.asarray(subject, dtype=np.float32), np.asarray(obj, dtype=np.float32)\n    for value in (s, o):\n        if value.shape != (4,) or not np.isfinite(value).all() or (value < 0).any() or (value > 1).any() or (value[2:] <= value[:2]).any():\n            raise ValueError(\'Invalid normalized xyxy box\')\n    return np.stack((s, o, np.r_[np.minimum(s[:2], o[:2]), np.maximum(s[2:], o[2:])])).tolist()\n\n\ndef human_pairs(cases, annotations, excluded, captions, relation_text):\n    result = []\n    for case in cases[\'cases\']:\n        if case[\'relation\'] not in (\'holding\', \'riding\'):\n            continue\n        a = annotations[\'annotations\'].get(case[\'case_id\'], {})\n        if case[\'filename\'] in excluded or captions[case[\'filename\']][\'split\'] not in (\'train\', \'restval\'):\n            continue\n        if any(a.get(key) != \'clear\' for key in (\'subject_box_status\', \'A_box_status\', \'B_box_status\')):\n            continue\n        status = (a.get(\'A_status\'), a.get(\'B_status\'))\n        if status not in ((\'supported\', \'contradicted\'), (\'contradicted\', \'supported\')):\n            continue\n        targets = case[\'targets\']\n        if targets[0][\'label\'] != targets[1][\'label\']:\n            continue\n        candidates = [dict(text=relation_text(case[\'relation\'], t[\'label\']), bounds=bounds(case[\'subject_box\'], t[\'bbox\'])) for t in targets]\n        result.append(dict(case_id=case[\'case_id\'], filename=case[\'filename\'], task=\'wrong_object_instance\',\n                           predicate=case[\'relation\'], object_label=targets[0][\'label\'], candidates=candidates,\n                           positive_index=status.index(\'supported\'), label_source=\'human_explicit_supported_vs_contradicted\'))\n    return result\n\n\ndef vcoco_pairs(records, excluded, captions, relation_text, per_predicate=50):\n    candidates = defaultdict(list)\n    # Restrict verb swaps to classes plausibly associated with riding; no phone-vs-horse shortcut.\n    ride_objects = {\'bicycle\', \'motorcycle\', \'horse\', \'elephant\', \'skateboard\', \'surfboard\', \'snowboard\', \'skis\', \'boat\', \'car\', \'bus\', \'truck\', \'train\'}\n    for image in records:\n        if image[\'filename\'] in excluded or captions[image[\'filename\']][\'split\'] not in (\'train\', \'restval\'):\n            continue\n        objects = {r[\'instance_id\']: r for r in image[\'instances\']}\n        labels = {(r[\'subject_id\'], r[\'predicate\']): r[\'label\'] for r in image[\'actor_action_labels\']}\n        for edge in image[\'relations\']:\n            sid, oid, verb = edge[\'subject_id\'], edge[\'object_id\'], edge[\'predicate\']\n            if verb not in (\'holding\', \'riding\'):\n                continue\n            subject, obj = objects[sid], objects[oid]\n            assert labels.get((sid, verb)) == 1 and subject[\'category\'] == \'person\'\n            b = bounds(subject[\'box\'], obj[\'box\'])\n            positive = dict(text=relation_text(verb, obj[\'category\']), bounds=b)\n            alternate = \'riding\' if verb == \'holding\' else \'holding\'\n            common = dict(filename=image[\'filename\'], predicate=verb, object_label=obj[\'category\'], label_source=\'vcoco_positive_role_and_explicit_actor_zero\')\n            if obj[\'category\'] in ride_objects and labels.get((sid, alternate)) == 0:\n                candidates[(\'wrong_predicate\', verb)].append(dict(common, case_id=f\'vcoco_verb_{image["image_id"]}_{sid}_{oid}_{verb}\',\n                    task=\'wrong_predicate\', candidates=[positive, dict(text=relation_text(alternate, obj[\'category\']), bounds=b)], positive_index=0))\n            for other in objects.values():\n                if other[\'instance_id\'] == sid or other[\'category\'] != \'person\' or labels.get((other[\'instance_id\'], verb)) != 0:\n                    continue\n                candidates[(\'wrong_subject_instance\', verb)].append(dict(common,\n                    case_id=f\'vcoco_actor_{image["image_id"]}_{sid}_{oid}_{other["instance_id"]}_{verb}\', task=\'wrong_subject_instance\',\n                    candidates=[positive, dict(text=positive[\'text\'], bounds=bounds(other[\'box\'], obj[\'box\']))], positive_index=0))\n    result = []\n    for task in (\'wrong_predicate\', \'wrong_subject_instance\'):\n        groups = []\n        for verb in (\'holding\', \'riding\'):\n            # One pair per image/predicate; selection depends only on stable IDs, never model scores.\n            ordered = sorted(candidates[(task, verb)], key=lambda c: identity(c[\'case_id\']))\n            seen, selected = set(), []\n            for row in ordered:\n                if row[\'filename\'] not in seen:\n                    seen.add(row[\'filename\']); selected.append(row)\n            groups.append(selected)\n        if task == \'wrong_predicate\':\n            # Balance positive verbs WITHIN each object class, before seeing any model score.\n            remaining = per_predicate\n            for label in sorted(ride_objects):\n                per_class = [[r for r in g if r[\'object_label\'] == label] for g in groups]\n                count = min(remaining, *(len(g) for g in per_class))\n                result.extend(row for group in per_class for row in group[:count])\n                remaining -= count\n        else:\n            count = min(per_predicate, *(len(g) for g in groups))\n            result.extend(row for group in groups for row in group[:count])\n    return result\n\n\ndef finalize(rows, captions):\n    result = []\n    for original in rows:\n        row = dict(original)\n        row[\'coco_id\'] = int(Path(row[\'filename\']).stem.rsplit(\'_\', 1)[1])\n        row[\'caption_split\'] = captions[row[\'filename\']][\'split\']\n        if int(identity(row[\'case_id\'])[:2], 16) % 2:\n            row[\'candidates\'] = row[\'candidates\'][::-1]\n            row[\'positive_index\'] = 1-row[\'positive_index\']\n        result.append(row)\n    assert len({r[\'case_id\'] for r in result}) == len(result)\n    return sorted(result, key=lambda r: (r[\'task\'], r[\'case_id\']))\n\n\ndef summarize(rows):\n    output = {}\n    for task in sorted({r[\'task\'] for r in rows}):\n        selected = [r for r in rows if r[\'task\'] == task]\n        methods = {}\n        for method in (\'boxed\', \'global_view\', \'zero_visual\'):\n            by_image = defaultdict(list)\n            margins, ties = [], 0\n            for row in selected:\n                positive = row[\'positive_index\']\n                scores = row[\'scores\'][method]\n                if len(scores) != 2 or not all(math.isfinite(s) for s in scores):\n                    raise ValueError(\'Nonfinite or incomplete scores\')\n                margin = scores[positive]-scores[1-positive]\n                tie = abs(margin) <= 1e-7\n                margins.append(margin); ties += tie\n                by_image[row[\'coco_id\']].append(.5 if tie else float(margin > 0))\n            values = np.asarray([np.mean(v) for v in by_image.values()])\n            rng = np.random.default_rng(20261019)\n            intervals = np.percentile([values[rng.integers(0, len(values), len(values))].mean() for _ in range(2000)], [2.5, 97.5])\n            methods[method] = dict(image_macro_pair_accuracy=float(values.mean()), image_cluster_bootstrap95=intervals.tolist(),\n                                   pair_accuracy=float(sum(sum(v) for v in by_image.values())/len(selected)),\n                                   ties=ties, mean_cosine_margin=float(np.mean(margins)))\n        output[task] = dict(pairs=len(selected), images=len({r[\'coco_id\'] for r in selected}),\n                            predicates=dict(Counter(r[\'predicate\'] for r in selected)), methods=methods)\n    return output\n', 'test_probe_data.py': "import unittest\nfrom probe_data import bounds, vcoco_pairs, human_pairs, summarize\n\n\ndef phrase(verb, obj): return f'a person {verb} a {obj}'\n\n\ndef image(name, verb, negative=0):\n    return dict(image_id=int(name), filename=name, instances=[\n        dict(instance_id=1,category='person',box=[0,0,.4,1]),\n        dict(instance_id=2,category='horse',box=[0,.4,1,1]),\n        dict(instance_id=3,category='person',box=[.6,0,1,1])],\n        relations=[dict(subject_id=1,object_id=2,predicate=verb)],\n        actor_action_labels=[dict(subject_id=1,predicate=verb,label=1),\n            dict(subject_id=1,predicate='riding' if verb=='holding' else 'holding',label=negative),\n            dict(subject_id=3,predicate=verb,label=negative)])\n\n\nclass ProbeDataChecks(unittest.TestCase):\n    def test_no_missing_or_both_positive_as_negative(self):\n        images=[image('1','holding',1), image('2','riding',1)]\n        captions={r['filename']:dict(split='train') for r in images}\n        self.assertEqual(vcoco_pairs(images,set(),captions,phrase),[])\n        for row in images: row['actor_action_labels']=row['actor_action_labels'][:1]\n        self.assertEqual(vcoco_pairs(images,set(),captions,phrase),[])\n\n    def test_balance_same_class_and_exclude_training(self):\n        images=[image('1','holding'),image('2','riding'),image('3','holding')]\n        captions={r['filename']:dict(split='train') for r in images}\n        rows=vcoco_pairs(images,{'3'},captions,phrase)\n        verbs=[r for r in rows if r['task']=='wrong_predicate']\n        self.assertEqual(len(verbs),2)\n        self.assertEqual({r['predicate'] for r in verbs},{'holding','riding'})\n        self.assertTrue(all(r['filename']!='3' for r in rows))\n        self.assertTrue(all(r['candidates'][0]['bounds']==r['candidates'][1]['bounds'] for r in verbs))\n\n    def test_human_requires_explicit_clear_contrast(self):\n        case=dict(case_id='x', filename='1', subject_box=[0,0,.4,1], relation='holding',\n                  targets=[dict(label='horse',bbox=[0,0,.5,1]),dict(label='horse',bbox=[.5,0,1,1])])\n        a=dict(subject_box_status='clear',A_box_status='clear',B_box_status='clear',A_status='supported',B_status='contradicted')\n        args=({'cases':[case]}, {'annotations':{'x':a}}, set(), {'1':{'split':'train'}}, phrase)\n        self.assertEqual(len(human_pairs(*args)),1)\n        a['B_status']='supported'\n        self.assertEqual(human_pairs(*args),[])\n        a['B_status']='contradicted';a['A_box_status']='incorrect'\n        self.assertEqual(human_pairs(*args),[])\n\n    def test_ties_are_half_credit_not_side_bias(self):\n        row=dict(task='wrong_predicate',predicate='holding',coco_id=1,positive_index=1,\n                 scores={k:[.2,.2] for k in ('boxed','global_view','zero_visual')})\n        result=summarize([row])['wrong_predicate']['methods']['boxed']\n        self.assertEqual(result['pair_accuracy'],.5)\n        self.assertEqual(result['ties'],1)\n\n    def test_invalid_box_rejected(self):\n        with self.assertRaises(ValueError): bounds([0,0,0,1],[0,0,1,1])\n\n\nif __name__=='__main__': unittest.main()\n", 'test_probe_model.py': '"""Synthetic CPU preflight only; this does not load the trained checkpoint."""\nimport unittest\nimport torch\nfrom pilot_model import EvidenceMemory\n\n\nclass ProbeModelChecks(unittest.TestCase):\n    def test_zero_visual_has_no_instance_preference(self):\n        torch.manual_seed(42)\n        model=EvidenceMemory().eval().requires_grad_(False)\n        text=torch.randn(1,512).expand(2,-1)\n        boxes=torch.tensor([[[0,0,.3,1],[.2,.2,.5,.8],[0,0,.5,1]],\n                            [[.7,0,1,1],[.5,.2,.8,.8],[.5,0,1,1]]],dtype=torch.float32)\n        with torch.inference_mode():\n            _,scores,_=model.attend(torch.zeros(2,196,768),text,torch.full((2,),2),boxes)\n        self.assertTrue(torch.isfinite(scores).all())\n        self.assertLess(abs(float(scores[0]-scores[1])),1e-7)\n        self.assertTrue(all(p.grad is None for p in model.parameters()))\n\n\nif __name__==\'__main__\': unittest.main()\n', 'pilot_model.py': '"""Text queries image values; no text residual in visual evidence or matching score."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass EvidenceMemory(nn.Module):\n    def __init__(self, dim=512):\n        super().__init__()\n        self.query = nn.Linear(512, dim)\n        self.role = nn.Embedding(3, dim)\n        self.key = nn.Linear(768, dim)\n        self.value = nn.Linear(768, dim)\n        self.position = nn.Linear(4, dim, bias=False)\n        self.norm = nn.LayerNorm(dim)\n        axis = (torch.arange(14, dtype=torch.float32) + .5) / 14\n        y, x = torch.meshgrid(axis, axis, indexing=\'ij\')\n        self.register_buffer(\'xy\', torch.stack((x.flatten(), y.flatten()), -1))\n\n    def attend(self, patches, text, roles, bounds):\n        # bounds: subject, object, allowed box; values contain image features only.\n        xy = self.xy[None].expand(len(text), -1, -1)\n        inside = (xy[:, None] >= bounds[:, :, None, :2]).all(-1) & (xy[:, None] <= bounds[:, :, None, 2:]).all(-1)\n        allowed = inside[:, 2].clone()\n        empty = ~allowed.any(-1)\n        center = (bounds[:, 2, :2] + bounds[:, 2, 2:]) / 2\n        nearest = ((xy-center[:, None])**2).sum(-1).argmin(-1)\n        allowed[torch.arange(len(text), device=text.device)[empty], nearest[empty]] = True\n        spatial = torch.cat((xy, inside[:, :2].transpose(1, 2).float()), -1)\n        q = self.query(text) + self.role(roles)\n        keys = self.key(patches) + self.position(spatial)\n        weights = ((keys*q[:, None]).sum(-1) / q.shape[-1]**.5).masked_fill(~allowed, -torch.inf).softmax(-1)\n        evidence = self.norm((weights[:, :, None]*self.value(patches)).sum(1))\n        score = (F.normalize(q, dim=-1)*F.normalize(evidence, dim=-1)).sum(-1)\n        return evidence, score, weights\n\n    def forward(self, patches, text, roles, valid, bounds):\n        shape = valid.shape\n        tokens, scores, attention = self.attend(patches.flatten(0,1), text.flatten(0,1), roles.flatten(), bounds.flatten(0,1))\n        return tokens.reshape(*shape, -1).masked_fill(~valid[..., None], 0), scores.reshape(shape), attention.reshape(*shape,196)\n\n    def alignment(self, patches, text, roles, valid, bounds, phrase_ids, image_ids, positive_scores):\n        """Local-rank pseudo-pair hinge loss; uncertain negatives are a stated limit."""\n        n, slots = valid.shape\n        words, role, ok = phrase_ids.flatten(), roles.flatten(), valid.flatten()\n        owner = torch.arange(n, device=text.device).repeat_interleave(slots)\n        # Exclude every exact phrase known anywhere in the target image, not just its positive slot.\n        known = ((words[None, :, None] == phrase_ids[:, None, :]) & valid[:, None, :]).any(-1)\n        eligible = ok[:, None] & ok[None, :] & (role[:, None] == role[None, :])\n        eligible &= image_ids[owner, None] != image_ids[owner][None, :]\n        eligible &= ~known[owner] & ~known[owner].T\n        has = eligible.any(-1)\n        donor = eligible.long().argmax(-1)\n        index = torch.nonzero(has, as_tuple=False).flatten()\n        if not len(index):\n            return positive_scores.sum()*0, dict(pairs=0, accuracy=None, margin=None)\n        donor = donor[index]\n        p, t, r, b = patches.flatten(0,1), text.flatten(0,1), roles.flatten(), bounds.flatten(0,1)\n        _, wrong_text, _ = self.attend(p[index], t[donor], r[index], b[index])\n        _, wrong_region, _ = self.attend(p[donor], t[index], r[index], b[donor])\n        positive = positive_scores.flatten()[index]\n        losses = (F.relu(.2-positive+wrong_text)+F.relu(.2-positive+wrong_region)) / 2\n        margin = torch.minimum(positive-wrong_text, positive-wrong_region)\n        return losses.mean(), dict(pairs=len(index), accuracy=float((margin>0).float().mean().detach()),\n                                   margin=float(margin.mean().detach()))\n\n\nclass PilotModel(nn.Module):\n    def __init__(self, base, arm):\n        super().__init__()\n        if arm not in (\'A\',\'E\'): raise ValueError(arm)\n        self.base, self.arm = base, arm\n        self.extra = EvidenceMemory() if arm==\'E\' else None\n        self.last_gates = None\n        self.last_alignment = dict(loss=0., pairs=0, accuracy=None, margin=None)\n\n    @property\n    def decoder(self): return self.base.decoder\n\n    def memory(self, batch, intervention=\'on\'):\n        memory, padding = self.base.build_memory(batch[\'visual\'], batch[\'legacy\'], batch[\'legacy_mask\'])\n        if self.extra is None: return memory, padding\n        shape = batch[\'valid\'].shape\n        patches = batch[\'visual\'][:, None, 1:].expand(-1, shape[1], -1, -1)\n        text, bounds = batch[\'text\'], batch[\'bounds\']\n        if intervention==\'text_swap\': text=batch[\'swap_text\']\n        elif intervention==\'region_swap\':\n            joint = batch[\'swap_valid\']\n            patches = torch.where(joint[...,None,None], batch[\'swap_visual\'][:,None,1:].expand_as(patches), patches)\n            bounds = torch.where(joint[...,None,None], batch[\'swap_bounds\'], bounds)\n        elif intervention not in (\'on\',\'off\'): raise ValueError(intervention)\n        tokens, scores, attention = self.extra(patches, text, batch[\'roles\'], batch[\'valid\'], bounds)\n        self.last_gates = torch.ones_like(scores)\n        self.last_attention = attention.detach()\n        self._evidence_inputs = (patches, text, batch[\'roles\'], batch[\'valid\'], bounds, scores)\n        extra_padding = ~batch[\'valid\']\n        if intervention==\'off\':\n            tokens=torch.zeros_like(tokens);extra_padding=torch.ones_like(extra_padding)\n        return torch.cat((memory,tokens),1), torch.cat((padding,extra_padding),1)\n\n    def teacher(self, memory, padding, targets, masks=None):\n        b, slots, dim = memory.shape\n        repeats = targets.size(0)//b\n        if repeats*b != targets.size(0): raise ValueError(\'Caption/image batch mismatch\')\n        m=memory[:,None].expand(b,repeats,slots,dim).reshape(-1,slots,dim)\n        p=padding[:,None].expand(b,repeats,slots).reshape(-1,slots)\n        return self.decoder(targets,m,tgt_key_padding_mask=masks,memory_key_padding_mask=p)\n\n    def forward(self, batch, sample, active, advantage, scst_weight, pad_idx):\n        from captioning.scst import policy_loss\n        memory,padding=self.memory(batch)\n        targets=batch[\'captions\'].flatten(0,1); masks=batch[\'caption_mask\'].flatten(0,1)\n        logits=self.teacher(memory,padding,targets[:,:-1],masks[:,:-1])\n        ce=F.cross_entropy(logits.flatten(0,1),targets[:,1:].reshape(-1),ignore_index=pad_idx)\n        rl=ce.new_zeros(())\n        if scst_weight:\n            rl=policy_loss(self.teacher(memory,padding,sample[:,:-1]),sample[:,1:],active,advantage)\n        alignment=ce.new_zeros(())\n        info=dict(pairs=0,accuracy=None,margin=None)\n        if self.extra is not None:\n            p,t,r,v,b,s=self._evidence_inputs\n            alignment,info=self.extra.alignment(p,t,r,v,b,batch[\'phrase_ids\'],batch[\'image_id\'],s)\n        self.last_alignment=dict(loss=float(alignment.detach()),**info)\n        return ce+scst_weight*rl+.1*alignment, ce.detach(), rl.detach()\n'}
SOURCE_HASHES = {'run_probe.py': 'd7f6f9cce38ed29a3d0b69f0ee84cce7390da069cd88fe6acd049b06963e9d81', 'probe_data.py': 'f8dfb24cf825ecc2935bfc062731009197459cd55ebb9b0cbb146d248d574e9f', 'test_probe_data.py': '2ad3d4e939658810daf4718dc9b176eb54b65f1a7c5ce147a47efb17a93095ea', 'test_probe_model.py': 'b6478609cbd49db09b4e1f90673c22925e7d508bb1ebf9947199d01e4f818cca', 'pilot_model.py': '4f8189de7d183bb062247043d6a2ce7bcb6fbd9fb7d0d485922c065d8cbe962b'}
MANIFEST_TEXT = '{\n  "schema_version": 1,\n  "purpose": "Frozen E matcher diagnostic; no caption training or checkpoint selection",\n  "checkpoint_sha256": "651f411f589c5dd7beb0fae98fafaa7d9ea06a7fd7f2b49400ac878f60227b18",\n  "checkpoint_protocol_sha256": "3e34a4561edc269152c0a61f3c76a807b56da85192a18d1e8bac99c8642a085f",\n  "checkpoint_cohort_sha256": "6bbdf3b75869927334ec6c2f868edf9f52346c0e79401aa0e118ba5a02fb711b",\n  "sources_sha256": {\n    "dataset": "2fd999220673258012acfb411a4e7e66af7d488050b2519b0badcc49b7600b8d",\n    "grounded": "7fa194c7651c348f92c5314e39940ae8249d3aef510fbe76f31edf1e2a61a45b",\n    "cases": "f009ad534f2c24033bb38f363662ce9012350229f48f2ff531b6733527f7ed96",\n    "review": "5ffdcedb7f9b72565f052b2e10860562c53ff9e1d4e386e6851e15291444507f"\n  },\n  "excluded_pilot_train_filenames_sha256": "7def3bf416b999aca519909c1d526df0d4fe756438604534ba187c995fcec5c0",\n  "excluded_pilot_train_images": 10000,\n  "test_used": false,\n  "validation_images_used": false,\n  "alignment_train_overlap": 0,\n  "caption_pretraining_overlap": "All diagnostic images are Karpathy train/restval, historically used for caption pretraining; not unseen-test evidence.",\n  "scope": "Existing reused human diagnostics + V-COCO val-role labels; GT boxes, not UPT detections; no annotation absence interpreted as false.",\n  "counts": {\n    "pairs": 62,\n    "images": 59,\n    "tasks": {\n      "wrong_object_instance": 12,\n      "wrong_predicate": 12,\n      "wrong_subject_instance": 38\n    },\n    "positive_predicates": {\n      "wrong_object_instance:riding": 9,\n      "wrong_object_instance:holding": 3,\n      "wrong_predicate:riding": 6,\n      "wrong_predicate:holding": 6,\n      "wrong_subject_instance:riding": 19,\n      "wrong_subject_instance:holding": 19\n    }\n  },\n  "rows": [\n    {\n      "case_id": "target_0c3f9530294c",\n      "filename": "COCO_train2014_000000402041.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "elephant",\n      "candidates": [\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.37229686975479126,\n              0.11458221077919006,\n              0.4605312645435333,\n              0.38474392890930176\n            ],\n            [\n              0.2417343705892563,\n              0.1992991864681244,\n              0.607796847820282,\n              0.8188948631286621\n            ],\n            [\n              0.2417343705892563,\n              0.11458221077919006,\n              0.607796847820282,\n              0.8188948631286621\n            ]\n          ]\n        },\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.37229686975479126,\n              0.11458221077919006,\n              0.4605312645435333,\n              0.38474392890930176\n            ],\n            [\n              0.03096874989569187,\n              0.18946091830730438,\n              0.3441718816757202,\n              0.8205660581588745\n            ],\n            [\n              0.03096874989569187,\n              0.11458221077919006,\n              0.4605312645435333,\n              0.8205660581588745\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 402041,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_0e4cbf68736f",\n      "filename": "COCO_train2014_000000329469.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.44103124737739563,\n              0.12934426963329315,\n              0.8724687695503235,\n              0.7125058770179749\n            ],\n            [\n              0.379031240940094,\n              0.4518032670021057,\n              0.9988593459129333,\n              0.9889227151870728\n            ],\n            [\n              0.379031240940094,\n              0.12934426963329315,\n              0.9988593459129333,\n              0.9889227151870728\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.44103124737739563,\n              0.12934426963329315,\n              0.8724687695503235,\n              0.7125058770179749\n            ],\n            [\n              0.00040625000838190317,\n              0.1662997603416443,\n              0.33406248688697815,\n              0.9707728624343872\n            ],\n            [\n              0.00040625000838190317,\n              0.12934426963329315,\n              0.8724687695503235,\n              0.9707728624343872\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 329469,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_1c641d5eb7b4",\n      "filename": "COCO_train2014_000000329469.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.054734375327825546,\n              0.002974238945171237,\n              0.3283750116825104,\n              0.6980562210083008\n            ],\n            [\n              0.379031240940094,\n              0.4518032670021057,\n              0.9988593459129333,\n              0.9889227151870728\n            ],\n            [\n              0.054734375327825546,\n              0.002974238945171237,\n              0.9988593459129333,\n              0.9889227151870728\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.054734375327825546,\n              0.002974238945171237,\n              0.3283750116825104,\n              0.6980562210083008\n            ],\n            [\n              0.00040625000838190317,\n              0.1662997603416443,\n              0.33406248688697815,\n              0.9707728624343872\n            ],\n            [\n              0.00040625000838190317,\n              0.002974238945171237,\n              0.33406248688697815,\n              0.9707728624343872\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 329469,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_4847bc6c9dbd",\n      "filename": "COCO_train2014_000000529613.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "holding",\n      "object_label": "pizza",\n      "candidates": [\n        {\n          "text": "a person holding a pizza",\n          "bounds": [\n            [\n              0.7104374766349792,\n              0.05479079484939575,\n              0.9990781545639038,\n              0.9891631603240967\n            ],\n            [\n              0.5698281526565552,\n              0.7525104880332947,\n              0.7062343955039978,\n              0.9071129560470581\n            ],\n            [\n              0.5698281526565552,\n              0.05479079484939575,\n              0.9990781545639038,\n              0.9891631603240967\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a pizza",\n          "bounds": [\n            [\n              0.7104374766349792,\n              0.05479079484939575,\n              0.9990781545639038,\n              0.9891631603240967\n            ],\n            [\n              0.7311093807220459,\n              0.5250208973884583,\n              0.8207187652587891,\n              0.6130962371826172\n            ],\n            [\n              0.7104374766349792,\n              0.05479079484939575,\n              0.9990781545639038,\n              0.9891631603240967\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 529613,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_a3e9de3d1a05",\n      "filename": "COCO_train2014_000000306800.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.12490624934434891,\n              0.07544626295566559,\n              0.2906718850135803,\n              0.4726047217845917\n            ],\n            [\n              0.12729687988758087,\n              0.2585063874721527,\n              0.3114687502384186,\n              0.5208925604820251\n            ],\n            [\n              0.12490624934434891,\n              0.07544626295566559,\n              0.3114687502384186,\n              0.5208925604820251\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.12490624934434891,\n              0.07544626295566559,\n              0.2906718850135803,\n              0.4726047217845917\n            ],\n            [\n              0.36570313572883606,\n              0.24967212975025177,\n              0.504281222820282,\n              0.5496357083320618\n            ],\n            [\n              0.12490624934434891,\n              0.07544626295566559,\n              0.504281222820282,\n              0.5496357083320618\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 306800,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_aad7419af8d7",\n      "filename": "COCO_train2014_000000090148.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "horse",\n      "candidates": [\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.44264063239097595,\n              0.4294103682041168,\n              0.5676562786102295,\n              0.828773558139801\n            ],\n            [\n              0.20103125274181366,\n              0.4789622724056244,\n              0.45512500405311584,\n              0.9908726215362549\n            ],\n            [\n              0.20103125274181366,\n              0.4294103682041168,\n              0.5676562786102295,\n              0.9908726215362549\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.44264063239097595,\n              0.4294103682041168,\n              0.5676562786102295,\n              0.828773558139801\n            ],\n            [\n              0.37553125619888306,\n              0.5447641611099243,\n              0.6722031235694885,\n              0.9727122783660889\n            ],\n            [\n              0.37553125619888306,\n              0.4294103682041168,\n              0.6722031235694885,\n              0.9727122783660889\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 90148,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_be7cb012482d",\n      "filename": "COCO_train2014_000000029525.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.5501093864440918,\n              0.5265630483627319,\n              0.5977500081062317,\n              0.6634003520011902\n            ],\n            [\n              0.7240312695503235,\n              0.5825777053833008,\n              0.779046893119812,\n              0.6706947088241577\n            ],\n            [\n              0.5501093864440918,\n              0.5265630483627319,\n              0.779046893119812,\n              0.6706947088241577\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.5501093864440918,\n              0.5265630483627319,\n              0.5977500081062317,\n              0.6634003520011902\n            ],\n            [\n              0.534250020980835,\n              0.5866178870201111,\n              0.6139531135559082,\n              0.6900548338890076\n            ],\n            [\n              0.534250020980835,\n              0.5265630483627319,\n              0.6139531135559082,\n              0.6900548338890076\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 29525,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_db610fa7def8",\n      "filename": "COCO_train2014_000000448839.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.6131406426429749,\n              0.27385416626930237,\n              0.7441562414169312,\n              0.7698749899864197\n            ],\n            [\n              0.3524687588214874,\n              0.4518125057220459,\n              0.48356249928474426,\n              0.7711041569709778\n            ],\n            [\n              0.3524687588214874,\n              0.27385416626930237,\n              0.7441562414169312,\n              0.7711041569709778\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.6131406426429749,\n              0.27385416626930237,\n              0.7441562414169312,\n              0.7698749899864197\n            ],\n            [\n              0.6295156478881836,\n              0.5354999899864197,\n              0.6914843916893005,\n              0.9014583230018616\n            ],\n            [\n              0.6131406426429749,\n              0.27385416626930237,\n              0.7441562414169312,\n              0.9014583230018616\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 448839,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_ddf0eb72e2b2",\n      "filename": "COCO_train2014_000000136302.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "holding",\n      "object_label": "surfboard",\n      "candidates": [\n        {\n          "text": "a person holding a surfboard",\n          "bounds": [\n            [\n              0.20231249928474426,\n              0.5519999861717224,\n              0.25632813572883606,\n              0.7668125033378601\n            ],\n            [\n              0.16318750381469727,\n              0.5120416879653931,\n              0.29820311069488525,\n              0.5669375061988831\n            ],\n            [\n              0.16318750381469727,\n              0.5120416879653931,\n              0.29820311069488525,\n              0.7668125033378601\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a surfboard",\n          "bounds": [\n            [\n              0.20231249928474426,\n              0.5519999861717224,\n              0.25632813572883606,\n              0.7668125033378601\n            ],\n            [\n              0.16573438048362732,\n              0.6129791736602783,\n              0.25764063000679016,\n              0.6891250014305115\n            ],\n            [\n              0.16573438048362732,\n              0.5519999861717224,\n              0.25764063000679016,\n              0.7668125033378601\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 136302,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_e0ba7ff124ef",\n      "filename": "COCO_train2014_000000334758.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "horse",\n      "candidates": [\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.24637646973133087,\n              0.5350156426429749,\n              0.4445176422595978,\n              0.7931249737739563\n            ],\n            [\n              0.047152940183877945,\n              0.6251093745231628,\n              0.2621176540851593,\n              0.9177968502044678\n            ],\n            [\n              0.047152940183877945,\n              0.5350156426429749,\n              0.4445176422595978,\n              0.9177968502044678\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.24637646973133087,\n              0.5350156426429749,\n              0.4445176422595978,\n              0.7931249737739563\n            ],\n            [\n              0.2870352864265442,\n              0.5880937576293945,\n              0.41312941908836365,\n              0.9177031517028809\n            ],\n            [\n              0.24637646973133087,\n              0.5350156426429749,\n              0.4445176422595978,\n              0.9177031517028809\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 334758,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_f4c47f24a715",\n      "filename": "COCO_train2014_000000432019.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "holding",\n      "object_label": "umbrella",\n      "candidates": [\n        {\n          "text": "a person holding an umbrella",\n          "bounds": [\n            [\n              0.6438906192779541,\n              0.8699444532394409,\n              0.7256718873977661,\n              0.9988611340522766\n            ],\n            [\n              0.5967968702316284,\n              0.7785555720329285,\n              0.8040156364440918,\n              0.9226111173629761\n            ],\n            [\n              0.5967968702316284,\n              0.7785555720329285,\n              0.8040156364440918,\n              0.9988611340522766\n            ]\n          ]\n        },\n        {\n          "text": "a person holding an umbrella",\n          "bounds": [\n            [\n              0.6438906192779541,\n              0.8699444532394409,\n              0.7256718873977661,\n              0.9988611340522766\n            ],\n            [\n              0.312624990940094,\n              0.7567777633666992,\n              0.5142655968666077,\n              0.9590277671813965\n            ],\n            [\n              0.312624990940094,\n              0.7567777633666992,\n              0.7256718873977661,\n              0.9988611340522766\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 432019,\n      "caption_split": "train"\n    },\n    {\n      "case_id": "target_fb525b4c45ce",\n      "filename": "COCO_train2014_000000402041.jpg",\n      "task": "wrong_object_instance",\n      "predicate": "riding",\n      "object_label": "elephant",\n      "candidates": [\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.6753281354904175,\n              0.2014285773038864,\n              0.772390604019165,\n              0.47886791825294495\n            ],\n            [\n              0.5250625014305115,\n              0.2986792325973511,\n              0.9226093888282776,\n              0.8185714483261108\n            ],\n            [\n              0.5250625014305115,\n              0.2014285773038864,\n              0.9226093888282776,\n              0.8185714483261108\n            ]\n          ]\n        },\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.6753281354904175,\n              0.2014285773038864,\n              0.772390604019165,\n              0.47886791825294495\n            ],\n            [\n              0.03096874989569187,\n              0.18946091830730438,\n              0.3441718816757202,\n              0.8205660581588745\n            ],\n            [\n              0.03096874989569187,\n              0.18946091830730438,\n              0.772390604019165,\n              0.8205660581588745\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "label_source": "human_explicit_supported_vs_contradicted",\n      "coco_id": 402041,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000116512.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_116512_217722_1765055_riding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.6652287840843201,\n              0.4274509847164154,\n              0.7954902052879333,\n              0.7481045722961426\n            ],\n            [\n              0.6561437845230103,\n              0.5800653696060181,\n              0.7348529696464539,\n              0.7608823776245117\n            ],\n            [\n              0.6561437845230103,\n              0.4274509847164154,\n              0.7954902052879333,\n              0.7608823776245117\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.6652287840843201,\n              0.4274509847164154,\n              0.7954902052879333,\n              0.7481045722961426\n            ],\n            [\n              0.6561437845230103,\n              0.5800653696060181,\n              0.7348529696464539,\n              0.7608823776245117\n            ],\n            [\n              0.6561437845230103,\n              0.4274509847164154,\n              0.7954902052879333,\n              0.7608823776245117\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 116512,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000117371.jpg",\n      "predicate": "holding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_117371_259686_129675_holding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.26661795377731323,\n              0.04307812452316284,\n              0.7634864449501038,\n              0.8140624761581421\n            ],\n            [\n              0.4051356911659241,\n              0.3806406259536743,\n              0.764321506023407,\n              0.982796847820282\n            ],\n            [\n              0.26661795377731323,\n              0.04307812452316284,\n              0.764321506023407,\n              0.982796847820282\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.26661795377731323,\n              0.04307812452316284,\n              0.7634864449501038,\n              0.8140624761581421\n            ],\n            [\n              0.4051356911659241,\n              0.3806406259536743,\n              0.764321506023407,\n              0.982796847820282\n            ],\n            [\n              0.26661795377731323,\n              0.04307812452316284,\n              0.764321506023407,\n              0.982796847820282\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 117371,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000167743.jpg",\n      "predicate": "holding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_167743_196203_127444_holding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.1712656319141388,\n              0.3935999870300293,\n              0.5667499899864197,\n              1.0\n            ],\n            [\n              0.5612343549728394,\n              0.9231294393539429,\n              0.6469218730926514,\n              0.9897882342338562\n            ],\n            [\n              0.1712656319141388,\n              0.3935999870300293,\n              0.6469218730926514,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.1712656319141388,\n              0.3935999870300293,\n              0.5667499899864197,\n              1.0\n            ],\n            [\n              0.5612343549728394,\n              0.9231294393539429,\n              0.6469218730926514,\n              0.9897882342338562\n            ],\n            [\n              0.1712656319141388,\n              0.3935999870300293,\n              0.6469218730926514,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 167743,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000231524.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_231524_234083_130532_riding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.7275624871253967,\n              0.23311474919319153,\n              0.9436718821525574,\n              1.0\n            ],\n            [\n              0.7545156478881836,\n              0.595714271068573,\n              1.0,\n              0.9862763285636902\n            ],\n            [\n              0.7275624871253967,\n              0.23311474919319153,\n              1.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.7275624871253967,\n              0.23311474919319153,\n              0.9436718821525574,\n              1.0\n            ],\n            [\n              0.7545156478881836,\n              0.595714271068573,\n              1.0,\n              0.9862763285636902\n            ],\n            [\n              0.7275624871253967,\n              0.23311474919319153,\n              1.0,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 231524,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000255495.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_255495_1716967_128488_riding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.1861562430858612,\n              0.0,\n              0.7047500014305115,\n              0.9887615442276001\n            ],\n            [\n              0.5949843525886536,\n              0.5667652487754822,\n              0.9813593626022339,\n              0.9895194172859192\n            ],\n            [\n              0.1861562430858612,\n              0.0,\n              0.9813593626022339,\n              0.9895194172859192\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.1861562430858612,\n              0.0,\n              0.7047500014305115,\n              0.9887615442276001\n            ],\n            [\n              0.5949843525886536,\n              0.5667652487754822,\n              0.9813593626022339,\n              0.9895194172859192\n            ],\n            [\n              0.1861562430858612,\n              0.0,\n              0.9813593626022339,\n              0.9895194172859192\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 255495,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000268313.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_268313_189084_128301_riding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.31773436069488525,\n              0.16183333098888397,\n              0.5048437714576721,\n              0.7833333611488342\n            ],\n            [\n              0.37257811427116394,\n              0.526520848274231,\n              0.4612968862056732,\n              0.8512499928474426\n            ],\n            [\n              0.31773436069488525,\n              0.16183333098888397,\n              0.5048437714576721,\n              0.8512499928474426\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.31773436069488525,\n              0.16183333098888397,\n              0.5048437714576721,\n              0.7833333611488342\n            ],\n            [\n              0.37257811427116394,\n              0.526520848274231,\n              0.4612968862056732,\n              0.8512499928474426\n            ],\n            [\n              0.31773436069488525,\n              0.16183333098888397,\n              0.5048437714576721,\n              0.8512499928474426\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 268313,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000340406.jpg",\n      "predicate": "holding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_340406_1265058_241798_holding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.8238281011581421,\n              0.4722500145435333,\n              0.9077187776565552,\n              0.8055833578109741\n            ],\n            [\n              0.8103437423706055,\n              0.648562490940094,\n              0.9169374704360962,\n              0.8288541436195374\n            ],\n            [\n              0.8103437423706055,\n              0.4722500145435333,\n              0.9169374704360962,\n              0.8288541436195374\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.8238281011581421,\n              0.4722500145435333,\n              0.9077187776565552,\n              0.8055833578109741\n            ],\n            [\n              0.8103437423706055,\n              0.648562490940094,\n              0.9169374704360962,\n              0.8288541436195374\n            ],\n            [\n              0.8103437423706055,\n              0.4722500145435333,\n              0.9169374704360962,\n              0.8288541436195374\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 340406,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000411238.jpg",\n      "predicate": "holding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_411238_254641_1334919_holding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.7264062762260437,\n              0.2202291637659073,\n              1.0,\n              0.9865208268165588\n            ],\n            [\n              0.7104843854904175,\n              0.8171666860580444,\n              0.8185312747955322,\n              0.984458327293396\n            ],\n            [\n              0.7104843854904175,\n              0.2202291637659073,\n              1.0,\n              0.9865208268165588\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.7264062762260437,\n              0.2202291637659073,\n              1.0,\n              0.9865208268165588\n            ],\n            [\n              0.7104843854904175,\n              0.8171666860580444,\n              0.8185312747955322,\n              0.984458327293396\n            ],\n            [\n              0.7104843854904175,\n              0.2202291637659073,\n              1.0,\n              0.9865208268165588\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 411238,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000459543.jpg",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_459543_185107_152411_riding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.45814061164855957,\n              0.3211241364479065,\n              0.7355937361717224,\n              0.5616159439086914\n            ],\n            [\n              0.5517968535423279,\n              0.5464637279510498,\n              0.8114687204360962,\n              0.7542856931686401\n            ],\n            [\n              0.45814061164855957,\n              0.3211241364479065,\n              0.8114687204360962,\n              0.7542856931686401\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a motorcycle",\n          "bounds": [\n            [\n              0.45814061164855957,\n              0.3211241364479065,\n              0.7355937361717224,\n              0.5616159439086914\n            ],\n            [\n              0.5517968535423279,\n              0.5464637279510498,\n              0.8114687204360962,\n              0.7542856931686401\n            ],\n            [\n              0.45814061164855957,\n              0.3211241364479065,\n              0.8114687204360962,\n              0.7542856931686401\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 459543,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000482810.jpg",\n      "predicate": "holding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_482810_197961_126737_holding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.3148750066757202,\n              0.14569412171840668,\n              0.5939218997955322,\n              0.9322117567062378\n            ],\n            [\n              0.024281250312924385,\n              0.5677411556243896,\n              0.711187481880188,\n              0.9784941077232361\n            ],\n            [\n              0.024281250312924385,\n              0.14569412171840668,\n              0.711187481880188,\n              0.9784941077232361\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.3148750066757202,\n              0.14569412171840668,\n              0.5939218997955322,\n              0.9322117567062378\n            ],\n            [\n              0.024281250312924385,\n              0.5677411556243896,\n              0.711187481880188,\n              0.9784941077232361\n            ],\n            [\n              0.024281250312924385,\n              0.14569412171840668,\n              0.711187481880188,\n              0.9784941077232361\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 482810,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000562296.jpg",\n      "predicate": "holding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_562296_195750_148370_holding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person holding a motorcycle",\n          "bounds": [\n            [\n              0.3037812411785126,\n              0.0015690376749262214,\n              0.8358281254768372,\n              0.9813389182090759\n            ],\n            [\n              0.001687500043772161,\n              0.3502301275730133,\n              1.0,\n              0.9898744821548462\n            ],\n            [\n              0.001687500043772161,\n              0.0015690376749262214,\n              1.0,\n              0.9898744821548462\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.3037812411785126,\n              0.0015690376749262214,\n              0.8358281254768372,\n              0.9813389182090759\n            ],\n            [\n              0.001687500043772161,\n              0.3502301275730133,\n              1.0,\n              0.9898744821548462\n            ],\n            [\n              0.001687500043772161,\n              0.0015690376749262214,\n              1.0,\n              0.9898744821548462\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 562296,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000098596.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_verb_98596_202874_125334_riding",\n      "task": "wrong_predicate",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.21579687297344208,\n              0.015435815788805485,\n              0.6891718506813049,\n              0.9950396418571472\n            ],\n            [\n              0.0,\n              0.6626624464988708,\n              0.8251405954360962,\n              1.0\n            ],\n            [\n              0.0,\n              0.015435815788805485,\n              0.8251405954360962,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a bicycle",\n          "bounds": [\n            [\n              0.21579687297344208,\n              0.015435815788805485,\n              0.6891718506813049,\n              0.9950396418571472\n            ],\n            [\n              0.0,\n              0.6626624464988708,\n              0.8251405954360962,\n              1.0\n            ],\n            [\n              0.0,\n              0.015435815788805485,\n              0.8251405954360962,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 98596,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000101094.jpg",\n      "predicate": "riding",\n      "object_label": "elephant",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_101094_1717869_583064_233930_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.40681248903274536,\n              0.15318749845027924,\n              0.4965781271457672,\n              0.29897916316986084\n            ],\n            [\n              0.19001562893390656,\n              0.3318333327770233,\n              0.6894687414169312,\n              0.8296041488647461\n            ],\n            [\n              0.19001562893390656,\n              0.15318749845027924,\n              0.6894687414169312,\n              0.8296041488647461\n            ]\n          ]\n        },\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.8354687690734863,\n              0.5525416731834412,\n              0.8940312266349792,\n              0.7758333086967468\n            ],\n            [\n              0.19001562893390656,\n              0.3318333327770233,\n              0.6894687414169312,\n              0.8296041488647461\n            ],\n            [\n              0.19001562893390656,\n              0.3318333327770233,\n              0.8940312266349792,\n              0.8296041488647461\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 101094,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000105714.jpg",\n      "predicate": "holding",\n      "object_label": "frisbee",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_105714_482561_602761_483544_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a frisbee",\n          "bounds": [\n            [\n              0.6565937399864197,\n              0.22222483158111572,\n              1.0,\n              0.9841217994689941\n            ],\n            [\n              0.3793906271457672,\n              0.3893676698207855,\n              0.44584375619888306,\n              0.5232552886009216\n            ],\n            [\n              0.3793906271457672,\n              0.22222483158111572,\n              1.0,\n              0.9841217994689941\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a frisbee",\n          "bounds": [\n            [\n              0.4212968647480011,\n              0.22177985310554504,\n              0.908578097820282,\n              0.9813348650932312\n            ],\n            [\n              0.3793906271457672,\n              0.3893676698207855,\n              0.44584375619888306,\n              0.5232552886009216\n            ],\n            [\n              0.3793906271457672,\n              0.22177985310554504,\n              0.908578097820282,\n              0.9813348650932312\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 105714,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000126001.jpg",\n      "predicate": "riding",\n      "object_label": "elephant",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_126001_563454_582401_559045_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.5190312266349792,\n              0.43960416316986084,\n              0.7751093506813049,\n              0.8023958206176758\n            ],\n            [\n              0.23310936987400055,\n              0.7961666584014893,\n              0.8969531059265137,\n              0.9898750185966492\n            ],\n            [\n              0.23310936987400055,\n              0.43960416316986084,\n              0.8969531059265137,\n              0.9898750185966492\n            ]\n          ]\n        },\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.32167187333106995,\n              0.5434583425521851,\n              0.4909687638282776,\n              0.9881458282470703\n            ],\n            [\n              0.23310936987400055,\n              0.7961666584014893,\n              0.8969531059265137,\n              0.9898750185966492\n            ],\n            [\n              0.23310936987400055,\n              0.5434583425521851,\n              0.8969531059265137,\n              0.9898750185966492\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 126001,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000147674.jpg",\n      "predicate": "holding",\n      "object_label": "handbag",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_147674_524611_1172736_2007716_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a handbag",\n          "bounds": [\n            [\n              0.0,\n              0.730772852897644,\n              0.1152968779206276,\n              1.0\n            ],\n            [\n              0.14671875536441803,\n              0.8357142806053162,\n              0.20200000703334808,\n              0.9436768293380737\n            ],\n            [\n              0.0,\n              0.730772852897644,\n              0.20200000703334808,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a handbag",\n          "bounds": [\n            [\n              0.13314062356948853,\n              0.7485246062278748,\n              0.2345000058412552,\n              0.9775409698486328\n            ],\n            [\n              0.14671875536441803,\n              0.8357142806053162,\n              0.20200000703334808,\n              0.9436768293380737\n            ],\n            [\n              0.13314062356948853,\n              0.7485246062278748,\n              0.2345000058412552,\n              0.9775409698486328\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 147674,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000159176.jpg",\n      "predicate": "holding",\n      "object_label": "skateboard",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_159176_1301495_1474767_1231990_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a skateboard",\n          "bounds": [\n            [\n              0.5081250071525574,\n              0.42853185534477234,\n              0.5763280987739563,\n              0.7341551184654236\n            ],\n            [\n              0.2615000009536743,\n              0.5813850164413452,\n              0.2896406352519989,\n              0.7102493047714233\n            ],\n            [\n              0.2615000009536743,\n              0.42853185534477234,\n              0.5763280987739563,\n              0.7341551184654236\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a skateboard",\n          "bounds": [\n            [\n              0.2423125058412552,\n              0.49980610609054565,\n              0.28987500071525574,\n              0.7234072089195251\n            ],\n            [\n              0.2615000009536743,\n              0.5813850164413452,\n              0.2896406352519989,\n              0.7102493047714233\n            ],\n            [\n              0.2423125058412552,\n              0.49980610609054565,\n              0.28987500071525574,\n              0.7234072089195251\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 159176,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000165334.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_165334_225612_1335895_217739_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.6270312666893005,\n              0.3283958435058594,\n              0.8047812581062317,\n              0.6598958373069763\n            ],\n            [\n              0.48310938477516174,\n              0.5619375109672546,\n              0.5456093549728394,\n              0.8479791879653931\n            ],\n            [\n              0.48310938477516174,\n              0.3283958435058594,\n              0.8047812581062317,\n              0.8479791879653931\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.44987499713897705,\n              0.3238750100135803,\n              0.5939218997955322,\n              0.7230833172798157\n            ],\n            [\n              0.48310938477516174,\n              0.5619375109672546,\n              0.5456093549728394,\n              0.8479791879653931\n            ],\n            [\n              0.44987499713897705,\n              0.3238750100135803,\n              0.5939218997955322,\n              0.8479791879653931\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 165334,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000170636.jpg",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_170636_219821_150628_236396_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.8752031326293945,\n              0.31016626954078674,\n              0.9940780997276306,\n              0.6931828856468201\n            ],\n            [\n              0.5203906297683716,\n              0.3354869484901428,\n              0.7796562314033508,\n              0.8285035490989685\n            ],\n            [\n              0.5203906297683716,\n              0.31016626954078674,\n              0.9940780997276306,\n              0.8285035490989685\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.5973437428474426,\n              0.266128271818161,\n              0.7292500138282776,\n              0.739406168460846\n            ],\n            [\n              0.5203906297683716,\n              0.3354869484901428,\n              0.7796562314033508,\n              0.8285035490989685\n            ],\n            [\n              0.5203906297683716,\n              0.266128271818161,\n              0.7796562314033508,\n              0.8285035490989685\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 170636,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000180778.jpg",\n      "predicate": "holding",\n      "object_label": "laptop",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_180778_1739914_1101257_454965_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a laptop",\n          "bounds": [\n            [\n              0.25120311975479126,\n              0.17098326981067657,\n              0.49904686212539673,\n              0.6585773825645447\n            ],\n            [\n              0.06456249952316284,\n              0.4424686133861542,\n              0.585281252861023,\n              0.8800627589225769\n            ],\n            [\n              0.06456249952316284,\n              0.17098326981067657,\n              0.585281252861023,\n              0.8800627589225769\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a laptop",\n          "bounds": [\n            [\n              0.0,\n              0.22715480625629425,\n              0.32087498903274536,\n              1.0\n            ],\n            [\n              0.06456249952316284,\n              0.4424686133861542,\n              0.585281252861023,\n              0.8800627589225769\n            ],\n            [\n              0.0,\n              0.22715480625629425,\n              0.585281252861023,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 180778,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000203671.jpg",\n      "predicate": "riding",\n      "object_label": "horse",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_203671_2161494_1403380_192903_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.8595468997955322,\n              0.1662295013666153,\n              1.0,\n              0.9870023131370544\n            ],\n            [\n              0.874093770980835,\n              0.5901405215263367,\n              0.9988124966621399,\n              0.997751772403717\n            ],\n            [\n              0.8595468997955322,\n              0.1662295013666153,\n              1.0,\n              0.997751772403717\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.4392968714237213,\n              0.5415691137313843,\n              0.725656270980835,\n              0.9887588024139404\n            ],\n            [\n              0.874093770980835,\n              0.5901405215263367,\n              0.9988124966621399,\n              0.997751772403717\n            ],\n            [\n              0.4392968714237213,\n              0.5415691137313843,\n              0.9988124966621399,\n              0.997751772403717\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 203671,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000209842.jpg",\n      "predicate": "riding",\n      "object_label": "horse",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_209842_188740_54707_205928_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.518750011920929,\n              0.4000000059604645,\n              0.6521875262260437,\n              0.8943793773651123\n            ],\n            [\n              0.2000156193971634,\n              0.44131147861480713,\n              0.45003125071525574,\n              0.9895082116127014\n            ],\n            [\n              0.2000156193971634,\n              0.4000000059604645,\n              0.6521875262260437,\n              0.9895082116127014\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.19757813215255737,\n              0.1193208396434784,\n              0.4896562397480011,\n              0.8832552433013916\n            ],\n            [\n              0.2000156193971634,\n              0.44131147861480713,\n              0.45003125071525574,\n              0.9895082116127014\n            ],\n            [\n              0.19757813215255737,\n              0.1193208396434784,\n              0.4896562397480011,\n              0.9895082116127014\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 209842,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000231315.jpg",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_231315_1252218_2204126_1275844_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.435734361410141,\n              0.0441787950694561,\n              0.7153280973434448,\n              0.889896035194397\n            ],\n            [\n              0.22248438000679016,\n              0.1283160150051117,\n              0.9484531283378601,\n              0.8904989361763\n            ],\n            [\n              0.22248438000679016,\n              0.0441787950694561,\n              0.9484531283378601,\n              0.8904989361763\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.01015624962747097,\n              0.0934719368815422,\n              0.3622343838214874,\n              0.9853638410568237\n            ],\n            [\n              0.22248438000679016,\n              0.1283160150051117,\n              0.9484531283378601,\n              0.8904989361763\n            ],\n            [\n              0.01015624962747097,\n              0.0934719368815422,\n              0.9484531283378601,\n              0.9853638410568237\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 231315,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000233901.jpg",\n      "predicate": "holding",\n      "object_label": "baseball glove",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_233901_458865_635536_435537_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a baseball glove",\n          "bounds": [\n            [\n              0.20884375274181366,\n              0.37626761198043823,\n              0.3518750071525574,\n              0.8165258169174194\n            ],\n            [\n              0.5512031316757202,\n              0.46718311309814453,\n              0.6017968654632568,\n              0.5823708772659302\n            ],\n            [\n              0.20884375274181366,\n              0.37626761198043823,\n              0.6017968654632568,\n              0.8165258169174194\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a baseball glove",\n          "bounds": [\n            [\n              0.4794843792915344,\n              0.3456572890281677,\n              0.8562343716621399,\n              0.8762206435203552\n            ],\n            [\n              0.5512031316757202,\n              0.46718311309814453,\n              0.6017968654632568,\n              0.5823708772659302\n            ],\n            [\n              0.4794843792915344,\n              0.3456572890281677,\n              0.8562343716621399,\n              0.8762206435203552\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 233901,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000268804.jpg",\n      "predicate": "riding",\n      "object_label": "horse",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_268804_1713522_54927_1739648_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.5056250095367432,\n              0.3685416579246521,\n              0.5882031321525574,\n              0.5415833592414856\n            ],\n            [\n              0.5102656483650208,\n              0.4934374988079071,\n              0.656906247138977,\n              0.7776874899864197\n            ],\n            [\n              0.5056250095367432,\n              0.3685416579246521,\n              0.656906247138977,\n              0.7776874899864197\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.6987344026565552,\n              0.5288958549499512,\n              0.7687187790870667,\n              0.6422291398048401\n            ],\n            [\n              0.5102656483650208,\n              0.4934374988079071,\n              0.656906247138977,\n              0.7776874899864197\n            ],\n            [\n              0.5102656483650208,\n              0.4934374988079071,\n              0.7687187790870667,\n              0.7776874899864197\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 268804,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000274267.jpg",\n      "predicate": "holding",\n      "object_label": "spoon",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_274267_1278180_706817_1682103_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a spoon",\n          "bounds": [\n            [\n              0.6503437757492065,\n              0.4178541600704193,\n              1.0,\n              0.7131458520889282\n            ],\n            [\n              0.22503125667572021,\n              0.5867083072662354,\n              0.2802656292915344,\n              0.6143749952316284\n            ],\n            [\n              0.22503125667572021,\n              0.4178541600704193,\n              1.0,\n              0.7131458520889282\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a spoon",\n          "bounds": [\n            [\n              0.10643749684095383,\n              0.3324166536331177,\n              0.3179374933242798,\n              0.633145809173584\n            ],\n            [\n              0.22503125667572021,\n              0.5867083072662354,\n              0.2802656292915344,\n              0.6143749952316284\n            ],\n            [\n              0.10643749684095383,\n              0.3324166536331177,\n              0.3179374933242798,\n              0.633145809173584\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 274267,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000027902.jpg",\n      "predicate": "holding",\n      "object_label": "sandwich",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_27902_2152824_2218436_450287_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a sandwich",\n          "bounds": [\n            [\n              0.28196653723716736,\n              0.2348593771457672,\n              0.9247907996177673,\n              0.9676406383514404\n            ],\n            [\n              0.5727823972702026,\n              0.5203593969345093,\n              0.6809204816818237,\n              0.5690000057220459\n            ],\n            [\n              0.28196653723716736,\n              0.2348593771457672,\n              0.9247907996177673,\n              0.9676406383514404\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a sandwich",\n          "bounds": [\n            [\n              0.8183891177177429,\n              0.10973437130451202,\n              1.0,\n              0.8647968769073486\n            ],\n            [\n              0.5727823972702026,\n              0.5203593969345093,\n              0.6809204816818237,\n              0.5690000057220459\n            ],\n            [\n              0.5727823972702026,\n              0.10973437130451202,\n              1.0,\n              0.8647968769073486\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 27902,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000284543.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_284543_187471_127037_2160149_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.32910001277923584,\n              0.12243243306875229,\n              0.49305999279022217,\n              0.3870870769023895\n            ],\n            [\n              0.24291999638080597,\n              0.4700600504875183,\n              0.42847999930381775,\n              0.8366066217422485\n            ],\n            [\n              0.24291999638080597,\n              0.12243243306875229,\n              0.49305999279022217,\n              0.8366066217422485\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.29934000968933105,\n              0.4795795679092407,\n              0.42824000120162964,\n              0.7569969892501831\n            ],\n            [\n              0.24291999638080597,\n              0.4700600504875183,\n              0.42847999930381775,\n              0.8366066217422485\n            ],\n            [\n              0.24291999638080597,\n              0.4700600504875183,\n              0.42847999930381775,\n              0.8366066217422485\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 284543,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000304445.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_304445_221015_124788_206773_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.6746562719345093,\n              0.3462603986263275,\n              0.7539218664169312,\n              0.7309695482254028\n            ],\n            [\n              0.2081875056028366,\n              0.532188355922699,\n              0.4018593728542328,\n              0.9270359873771667\n            ],\n            [\n              0.2081875056028366,\n              0.3462603986263275,\n              0.7539218664169312,\n              0.9270359873771667\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.2630000114440918,\n              0.23828254640102386,\n              0.408265620470047,\n              0.880166232585907\n            ],\n            [\n              0.2081875056028366,\n              0.532188355922699,\n              0.4018593728542328,\n              0.9270359873771667\n            ],\n            [\n              0.2081875056028366,\n              0.23828254640102386,\n              0.408265620470047,\n              0.9270359873771667\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 304445,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000304866.jpg",\n      "predicate": "holding",\n      "object_label": "backpack",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_304866_1249970_1169166_1254857_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a backpack",\n          "bounds": [\n            [\n              0.8590937256813049,\n              0.7146135568618774,\n              1.0,\n              0.9887588024139404\n            ],\n            [\n              0.726187527179718,\n              0.7536534070968628,\n              0.7729062438011169,\n              0.8705855011940002\n            ],\n            [\n              0.726187527179718,\n              0.7146135568618774,\n              1.0,\n              0.9887588024139404\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a backpack",\n          "bounds": [\n            [\n              0.7341406345367432,\n              0.6760655641555786,\n              0.8307656049728394,\n              1.0\n            ],\n            [\n              0.726187527179718,\n              0.7536534070968628,\n              0.7729062438011169,\n              0.8705855011940002\n            ],\n            [\n              0.726187527179718,\n              0.6760655641555786,\n              0.8307656049728394,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 304866,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000317306.jpg",\n      "predicate": "holding",\n      "object_label": "baseball bat",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_317306_462594_630774_437359_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a baseball bat",\n          "bounds": [\n            [\n              0.10923437774181366,\n              0.04030245915055275,\n              0.3489062488079071,\n              0.5273156762123108\n            ],\n            [\n              0.33107811212539673,\n              0.27202269434928894,\n              0.3860468864440918,\n              0.43829867243766785\n            ],\n            [\n              0.10923437774181366,\n              0.04030245915055275,\n              0.3860468864440918,\n              0.5273156762123108\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a baseball bat",\n          "bounds": [\n            [\n              0.6426718831062317,\n              0.24718336760997772,\n              0.8191249966621399,\n              0.8337051272392273\n            ],\n            [\n              0.33107811212539673,\n              0.27202269434928894,\n              0.3860468864440918,\n              0.43829867243766785\n            ],\n            [\n              0.33107811212539673,\n              0.24718336760997772,\n              0.8191249966621399,\n              0.8337051272392273\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 317306,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000356015.jpg",\n      "predicate": "holding",\n      "object_label": "horse",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_356015_1257852_274048_1328311_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a horse",\n          "bounds": [\n            [\n              0.24833999574184418,\n              0.40259701013565063,\n              0.3918600082397461,\n              0.6481492519378662\n            ],\n            [\n              0.5688999891281128,\n              0.4605970084667206,\n              0.6986799836158752,\n              0.8277015089988708\n            ],\n            [\n              0.24833999574184418,\n              0.40259701013565063,\n              0.6986799836158752,\n              0.8277015089988708\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a horse",\n          "bounds": [\n            [\n              0.5841799974441528,\n              0.37077611684799194,\n              0.6880599856376648,\n              0.6831343173980713\n            ],\n            [\n              0.5688999891281128,\n              0.4605970084667206,\n              0.6986799836158752,\n              0.8277015089988708\n            ],\n            [\n              0.5688999891281128,\n              0.37077611684799194,\n              0.6986799836158752,\n              0.8277015089988708\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 356015,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000384626.jpg",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_384626_2166601_151557_219924_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.1306000053882599,\n              0.3088372051715851,\n              0.5370200276374817,\n              0.8507493734359741\n            ],\n            [\n              0.0,\n              0.4883979260921478,\n              0.5101600289344788,\n              0.8718087673187256\n            ],\n            [\n              0.0,\n              0.3088372051715851,\n              0.5370200276374817,\n              0.8718087673187256\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.0,\n              0.002093023154884577,\n              0.09672000259160995,\n              0.4489147365093231\n            ],\n            [\n              0.0,\n              0.4883979260921478,\n              0.5101600289344788,\n              0.8718087673187256\n            ],\n            [\n              0.0,\n              0.002093023154884577,\n              0.5101600289344788,\n              0.8718087673187256\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 384626,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000407779.jpg",\n      "predicate": "holding",\n      "object_label": "teddy bear",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_407779_184993_1163135_213130_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a teddy bear",\n          "bounds": [\n            [\n              0.10090625286102295,\n              0.3345282971858978,\n              0.20973438024520874,\n              0.8309905529022217\n            ],\n            [\n              0.7837968468666077,\n              0.5545990467071533,\n              0.8712812662124634,\n              0.7482547163963318\n            ],\n            [\n              0.10090625286102295,\n              0.3345282971858978,\n              0.8712812662124634,\n              0.8309905529022217\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a teddy bear",\n          "bounds": [\n            [\n              0.703249990940094,\n              0.3645518720149994,\n              0.8288437724113464,\n              0.8353065848350525\n            ],\n            [\n              0.7837968468666077,\n              0.5545990467071533,\n              0.8712812662124634,\n              0.7482547163963318\n            ],\n            [\n              0.703249990940094,\n              0.3645518720149994,\n              0.8712812662124634,\n              0.8353065848350525\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 407779,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000421840.jpg",\n      "predicate": "holding",\n      "object_label": "person",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_421840_494406_506246_2206965_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a person",\n          "bounds": [\n            [\n              0.0,\n              0.36423438787460327,\n              0.41433027386665344,\n              0.7944999933242798\n            ],\n            [\n              0.3844220042228699,\n              0.3788124918937683,\n              0.9715780019760132,\n              0.8631093502044678\n            ],\n            [\n              0.0,\n              0.36423438787460327,\n              0.9715780019760132,\n              0.8631093502044678\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a person",\n          "bounds": [\n            [\n              0.43541285395622253,\n              0.08764062821865082,\n              0.9631926417350769,\n              0.6943749785423279\n            ],\n            [\n              0.3844220042228699,\n              0.3788124918937683,\n              0.9715780019760132,\n              0.8631093502044678\n            ],\n            [\n              0.3844220042228699,\n              0.08764062821865082,\n              0.9715780019760132,\n              0.8631093502044678\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 421840,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000445495.jpg",\n      "predicate": "holding",\n      "object_label": "book",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_445495_219382_1648592_229552_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a book",\n          "bounds": [\n            [\n              0.0042500002309679985,\n              0.24942928552627563,\n              0.47968751192092896,\n              0.9348387122154236\n            ],\n            [\n              0.34056249260902405,\n              0.3237965404987335,\n              0.4603906273841858,\n              0.6244168877601624\n            ],\n            [\n              0.0042500002309679985,\n              0.24942928552627563,\n              0.47968751192092896,\n              0.9348387122154236\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a book",\n          "bounds": [\n            [\n              0.1743437498807907,\n              0.23230768740177155,\n              0.8647500276565552,\n              0.8135980367660522\n            ],\n            [\n              0.34056249260902405,\n              0.3237965404987335,\n              0.4603906273841858,\n              0.6244168877601624\n            ],\n            [\n              0.1743437498807907,\n              0.23230768740177155,\n              0.8647500276565552,\n              0.8135980367660522\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 445495,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000453968.jpg",\n      "predicate": "holding",\n      "object_label": "tennis racket",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_453968_520651_1482094_539754_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a tennis racket",\n          "bounds": [\n            [\n              0.8533124923706055,\n              0.23717132210731506,\n              0.9932812452316284,\n              0.8383864760398865\n            ],\n            [\n              0.6481562256813049,\n              0.3039242923259735,\n              0.7539530992507935,\n              0.5649800896644592\n            ],\n            [\n              0.6481562256813049,\n              0.23717132210731506,\n              0.9932812452316284,\n              0.8383864760398865\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a tennis racket",\n          "bounds": [\n            [\n              0.5517030954360962,\n              0.1640438288450241,\n              0.7649843692779541,\n              0.9573107361793518\n            ],\n            [\n              0.6481562256813049,\n              0.3039242923259735,\n              0.7539530992507935,\n              0.5649800896644592\n            ],\n            [\n              0.5517030954360962,\n              0.1640438288450241,\n              0.7649843692779541,\n              0.9573107361793518\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 453968,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000456245.jpg",\n      "predicate": "riding",\n      "object_label": "boat",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_456245_201734_177098_213452_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a boat",\n          "bounds": [\n            [\n              0.47514063119888306,\n              0.5089976787567139,\n              0.6184531450271606,\n              0.7161072492599487\n            ],\n            [\n              0.3027656376361847,\n              0.6382051110267639,\n              0.8420312404632568,\n              0.7730302810668945\n            ],\n            [\n              0.3027656376361847,\n              0.5089976787567139,\n              0.8420312404632568,\n              0.7730302810668945\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a boat",\n          "bounds": [\n            [\n              0.5741093754768372,\n              0.4980885684490204,\n              0.6578906178474426,\n              0.7168298363685608\n            ],\n            [\n              0.3027656376361847,\n              0.6382051110267639,\n              0.8420312404632568,\n              0.7730302810668945\n            ],\n            [\n              0.3027656376361847,\n              0.4980885684490204,\n              0.8420312404632568,\n              0.7730302810668945\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 456245,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000457011.jpg",\n      "predicate": "riding",\n      "object_label": "horse",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_457011_230651_58352_234138_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.3334687352180481,\n              0.11304166913032532,\n              0.47556251287460327,\n              0.5081250071525574\n            ],\n            [\n              0.15517187118530273,\n              0.2737083435058594,\n              0.6514062285423279,\n              0.7844791412353516\n            ],\n            [\n              0.15517187118530273,\n              0.11304166913032532,\n              0.6514062285423279,\n              0.7844791412353516\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a horse",\n          "bounds": [\n            [\n              0.7066875100135803,\n              0.31860417127609253,\n              0.8377968668937683,\n              0.780916690826416\n            ],\n            [\n              0.15517187118530273,\n              0.2737083435058594,\n              0.6514062285423279,\n              0.7844791412353516\n            ],\n            [\n              0.15517187118530273,\n              0.2737083435058594,\n              0.8377968668937683,\n              0.7844791412353516\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 457011,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000464888.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_464888_200122_125247_256669_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.5639328360557556,\n              0.27248436212539673,\n              0.6367786526679993,\n              0.42353126406669617\n            ],\n            [\n              0.02442687749862671,\n              0.6890937685966492,\n              0.2849999964237213,\n              0.8178437352180481\n            ],\n            [\n              0.02442687749862671,\n              0.27248436212539673,\n              0.6367786526679993,\n              0.8178437352180481\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.10278656333684921,\n              0.5953750014305115,\n              0.2369762808084488,\n              0.7872499823570251\n            ],\n            [\n              0.02442687749862671,\n              0.6890937685966492,\n              0.2849999964237213,\n              0.8178437352180481\n            ],\n            [\n              0.02442687749862671,\n              0.5953750014305115,\n              0.2849999964237213,\n              0.8178437352180481\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 464888,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000464888.jpg",\n      "predicate": "holding",\n      "object_label": "suitcase",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_464888_223869_1186932_190039_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a suitcase",\n          "bounds": [\n            [\n              0.5904347896575928,\n              0.5920156240463257,\n              0.6591106653213501,\n              0.7785624861717224\n            ],\n            [\n              0.45915019512176514,\n              0.6760937571525574,\n              0.5139525532722473,\n              0.7692343592643738\n            ],\n            [\n              0.45915019512176514,\n              0.5920156240463257,\n              0.6591106653213501,\n              0.7785624861717224\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a suitcase",\n          "bounds": [\n            [\n              0.4890909194946289,\n              0.5966249704360962,\n              0.5853161811828613,\n              0.7798905968666077\n            ],\n            [\n              0.45915019512176514,\n              0.6760937571525574,\n              0.5139525532722473,\n              0.7692343592643738\n            ],\n            [\n              0.45915019512176514,\n              0.5966249704360962,\n              0.5853161811828613,\n              0.7798905968666077\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 464888,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000474190.jpg",\n      "predicate": "holding",\n      "object_label": "person",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_474190_512370_482591_482591_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a person",\n          "bounds": [\n            [\n              0.43229687213897705,\n              0.36110416054725647,\n              0.555343747138977,\n              0.6881874799728394\n            ],\n            [\n              0.466859370470047,\n              0.4606666564941406,\n              0.5258437395095825,\n              0.6921250224113464\n            ],\n            [\n              0.43229687213897705,\n              0.36110416054725647,\n              0.555343747138977,\n              0.6921250224113464\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a person",\n          "bounds": [\n            [\n              0.466859370470047,\n              0.4606666564941406,\n              0.5258437395095825,\n              0.6921250224113464\n            ],\n            [\n              0.466859370470047,\n              0.4606666564941406,\n              0.5258437395095825,\n              0.6921250224113464\n            ],\n            [\n              0.466859370470047,\n              0.4606666564941406,\n              0.5258437395095825,\n              0.6921250224113464\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 474190,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000481036.jpg",\n      "predicate": "riding",\n      "object_label": "train",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_481036_1237019_268194_1275881_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a train",\n          "bounds": [\n            [\n              0.44089582562446594,\n              0.3227187395095825,\n              0.6310416460037231,\n              0.5555468797683716\n            ],\n            [\n              0.0,\n              0.0650937482714653,\n              0.7427291870117188,\n              0.9890312552452087\n            ],\n            [\n              0.0,\n              0.0650937482714653,\n              0.7427291870117188,\n              0.9890312552452087\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a train",\n          "bounds": [\n            [\n              0.8748124837875366,\n              0.14228124916553497,\n              0.9995833039283752,\n              0.45720311999320984\n            ],\n            [\n              0.0,\n              0.0650937482714653,\n              0.7427291870117188,\n              0.9890312552452087\n            ],\n            [\n              0.0,\n              0.0650937482714653,\n              0.9995833039283752,\n              0.9890312552452087\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 481036,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000502756.jpg",\n      "predicate": "holding",\n      "object_label": "frisbee",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_502756_442481_604412_474343_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a frisbee",\n          "bounds": [\n            [\n              0.0,\n              0.4480421543121338,\n              0.23180000483989716,\n              1.0\n            ],\n            [\n              0.4429999887943268,\n              0.5345180630683899,\n              0.5017600059509277,\n              0.624518096446991\n            ],\n            [\n              0.0,\n              0.4480421543121338,\n              0.5017600059509277,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a frisbee",\n          "bounds": [\n            [\n              0.4032000005245209,\n              0.3403012156486511,\n              0.5485799908638,\n              0.8865662813186646\n            ],\n            [\n              0.4429999887943268,\n              0.5345180630683899,\n              0.5017600059509277,\n              0.624518096446991\n            ],\n            [\n              0.4032000005245209,\n              0.3403012156486511,\n              0.5485799908638,\n              0.8865662813186646\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 502756,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000517485.jpg",\n      "predicate": "riding",\n      "object_label": "bicycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_517485_2151314_126923_220994_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.23281249403953552,\n              0.24875587224960327,\n              0.41340625286102295,\n              0.8048121929168701\n            ],\n            [\n              0.4263593852519989,\n              0.4787324070930481,\n              0.974359393119812,\n              1.0\n            ],\n            [\n              0.23281249403953552,\n              0.24875587224960327,\n              0.974359393119812,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a bicycle",\n          "bounds": [\n            [\n              0.6464999914169312,\n              0.18669013679027557,\n              1.0,\n              0.9361032843589783\n            ],\n            [\n              0.4263593852519989,\n              0.4787324070930481,\n              0.974359393119812,\n              1.0\n            ],\n            [\n              0.4263593852519989,\n              0.18669013679027557,\n              1.0,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 517485,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000519053.jpg",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_519053_1255229_151303_1315996_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.4609687626361847,\n              0.19868852198123932,\n              0.5960624814033508,\n              0.5819672346115112\n            ],\n            [\n              0.38967186212539673,\n              0.2789227068424225,\n              0.6225937604904175,\n              0.7133255004882812\n            ],\n            [\n              0.38967186212539673,\n              0.19868852198123932,\n              0.6225937604904175,\n              0.7133255004882812\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.04284375160932541,\n              0.26208430528640747,\n              0.12017187476158142,\n              0.633583128452301\n            ],\n            [\n              0.38967186212539673,\n              0.2789227068424225,\n              0.6225937604904175,\n              0.7133255004882812\n            ],\n            [\n              0.04284375160932541,\n              0.26208430528640747,\n              0.6225937604904175,\n              0.7133255004882812\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 519053,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000542450.jpg",\n      "predicate": "riding",\n      "object_label": "motorcycle",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_542450_222035_149273_238770_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.3241249918937683,\n              0.4592037498950958,\n              0.4429062604904175,\n              0.7897658348083496\n            ],\n            [\n              0.3343437612056732,\n              0.557564377784729,\n              0.4557812511920929,\n              0.8519437909126282\n            ],\n            [\n              0.3241249918937683,\n              0.4592037498950958,\n              0.4557812511920929,\n              0.8519437909126282\n            ]\n          ]\n        },\n        {\n          "text": "a person riding a motorcycle",\n          "bounds": [\n            [\n              0.6363750100135803,\n              0.49852457642555237,\n              0.7122499942779541,\n              0.9100233912467957\n            ],\n            [\n              0.3343437612056732,\n              0.557564377784729,\n              0.4557812511920929,\n              0.8519437909126282\n            ],\n            [\n              0.3343437612056732,\n              0.49852457642555237,\n              0.7122499942779541,\n              0.9100233912467957\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 542450,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000566245.jpg",\n      "predicate": "riding",\n      "object_label": "elephant",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_566245_490789_580783_1231374_riding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.8583199977874756,\n              0.5258399844169617,\n              1.0,\n              0.9929199814796448\n            ],\n            [\n              0.041039999574422836,\n              0.5013800263404846,\n              0.8638666868209839,\n              1.0\n            ],\n            [\n              0.041039999574422836,\n              0.5013800263404846,\n              1.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          "text": "a person riding an elephant",\n          "bounds": [\n            [\n              0.07077333331108093,\n              0.12849999964237213,\n              0.4320266544818878,\n              0.6620200276374817\n            ],\n            [\n              0.041039999574422836,\n              0.5013800263404846,\n              0.8638666868209839,\n              1.0\n            ],\n            [\n              0.041039999574422836,\n              0.12849999964237213,\n              0.8638666868209839,\n              1.0\n            ]\n          ]\n        }\n      ],\n      "positive_index": 1,\n      "coco_id": 566245,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000068459.jpg",\n      "predicate": "holding",\n      "object_label": "tennis racket",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_68459_441167_657426_457917_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a tennis racket",\n          "bounds": [\n            [\n              0.19895312190055847,\n              0.15716665983200073,\n              0.3883593678474426,\n              0.7863749861717224\n            ],\n            [\n              0.23720312118530273,\n              0.11820833384990692,\n              0.2933281362056732,\n              0.25458332896232605\n            ],\n            [\n              0.19895312190055847,\n              0.11820833384990692,\n              0.3883593678474426,\n              0.7863749861717224\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a tennis racket",\n          "bounds": [\n            [\n              0.03700000047683716,\n              0.1300416737794876,\n              0.22870312631130219,\n              0.6031458377838135\n            ],\n            [\n              0.23720312118530273,\n              0.11820833384990692,\n              0.2933281362056732,\n              0.25458332896232605\n            ],\n            [\n              0.03700000047683716,\n              0.11820833384990692,\n              0.2933281362056732,\n              0.6031458377838135\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 68459,\n      "caption_split": "train"\n    },\n    {\n      "filename": "COCO_train2014_000000073903.jpg",\n      "predicate": "holding",\n      "object_label": "frisbee",\n      "label_source": "vcoco_positive_role_and_explicit_actor_zero",\n      "case_id": "vcoco_actor_73903_1188939_603900_1716867_holding",\n      "task": "wrong_subject_instance",\n      "candidates": [\n        {\n          "text": "a person holding a frisbee",\n          "bounds": [\n            [\n              0.6294353008270264,\n              0.21123437583446503,\n              0.8358588218688965,\n              0.9011250138282776\n            ],\n            [\n              0.7537882328033447,\n              0.11801562458276749,\n              0.872494101524353,\n              0.16107812523841858\n            ],\n            [\n              0.6294353008270264,\n              0.11801562458276749,\n              0.872494101524353,\n              0.9011250138282776\n            ]\n          ]\n        },\n        {\n          "text": "a person holding a frisbee",\n          "bounds": [\n            [\n              0.12207058817148209,\n              0.5058281421661377,\n              0.4531058967113495,\n              0.9892656207084656\n            ],\n            [\n              0.7537882328033447,\n              0.11801562458276749,\n              0.872494101524353,\n              0.16107812523841858\n            ],\n            [\n              0.12207058817148209,\n              0.11801562458276749,\n              0.872494101524353,\n              0.9892656207084656\n            ]\n          ]\n        }\n      ],\n      "positive_index": 0,\n      "coco_id": 73903,\n      "caption_split": "train"\n    }\n  ],\n  "rows_sha256": "d6f95949cccc476ee4bfbcff37cfdccc756db83d5a4957ecffaf45d251ca16b3"\n}'
for name, source in SOURCES.items(): (PROBE/name).write_text(source,encoding='utf-8')
MANIFEST = PROBE/'probe_manifest.json'; MANIFEST.write_text(MANIFEST_TEXT,encoding='utf-8')
for name, expected in SOURCE_HASHES.items(): assert hashlib.sha256((PROBE/name).read_bytes()).hexdigest()==expected


In [ ]:
import torch
assert torch.cuda.device_count()==2, 'Select GPU T4 x2.'
for rank in (0,1): print('GPU',rank,torch.cuda.get_device_name(rank),flush=True)
run([sys.executable,'-m','unittest','test_probe_data','test_probe_model','-v'],cwd=PROBE)
run([sys.executable,'-u','-c',"from transformers import CLIPTextModel,AutoTokenizer; r='57c216476eefef5ab752ec549e440a49ae4ae5f3'; CLIPTextModel.from_pretrained('openai/clip-vit-base-patch16',revision=r); AutoTokenizer.from_pretrained('openai/clip-vit-base-patch16',revision=r); print('Pinned text encoder ready',flush=True)"])
def sha(path):
    value=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''): value.update(block)
    return value.hexdigest()
expected=json.loads(MANIFEST_TEXT)['checkpoint_sha256']
assert sha(CHECKPOINT)==expected, 'Checkpoint does not match completed E. Do not substitute original/A.'
cfg=dict(checkpoint=str(CHECKPOINT),visual_cache=str(VISUAL_CACHE),output=str(OUTPUT),
         manifest=str(MANIFEST),manifest_sha256=sha(MANIFEST),source_sha256=SOURCE_HASHES)
CONFIG=PROBE/'config.json'; CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
run([sys.executable,'-u',PROBE/'run_probe.py','--config',CONFIG],cwd=REPO)


## Đọc kết quả
Tải `semantic_evidence_matcher_probe_review.zip` gửi lại. ZIP không chứa checkpoint.
`summary.json` báo tỷ lệ chọn cặp đúng, hòa tính0.5; bootstrap theo ảnh. Điểm cosine không phải xác suất.
So `boxed` với `global_view` (bỏ ràng buộc box) và `zero_visual` (xóa đặc trưng ảnh).
Nhãn âm là người đã chấm hoặc V-COCO ghi rõ actor/action=0; không suy từ thiếu nhãn.
Cặp đổi động từ cân bằng holding/riding trong từng lớp vật thể; vẫn chỉ12 cặp nên không kết luận rộng.
Điểm tốt trên bộ này cũng chưa chứng minh tăng CIDEr hoặc tính đúng của caption.